# 03 — Prepare ohsome API boundaries

This notebook prepares the validated 1 km Petrópolis grid for subsequent queries to the **ohsome API**.

It does **not** perform bulk API requests. Its purpose is to:

1. load and validate `data/processed/Petropolis/petropolis_grid_1km.parquet`;
2. preserve `grid_1km_id` as the permanent spatial key;
3. convert a query copy of the full 1 km cell geometries from SIRGAS 2000 (EPSG:4674) to WGS 84 (EPSG:4326);
4. prepare an identified GeoJSON `FeatureCollection` suitable for the ohsome API `bpolys` parameter and `groupBy/boundary` endpoints;
5. validate geometry, coordinate ranges, payload size, and identifiers;
6. create a deterministic 10-cell test subset for the next notebook.

The full 1 km cell geometries are retained. They are **not clipped to the Petrópolis boundary**, which keeps every analytical unit equal to approximately 1 km².

Official ohsome API documentation used for the boundary representation:

- https://docs.ohsome.org/ohsome-api/stable/group-by.html
- https://docs.ohsome.org/ohsome-api/development/boundaries.html

The ohsome API expects WGS 84 coordinates in longitude-latitude order. GeoJSON polygon boundaries can be submitted through the `bpolys` parameter, and an `id` property can be used to identify each boundary in grouped responses.


In [1]:
from pathlib import Path
import json
import sys
import platform

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import mapping

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("GeoPandas:", gpd.__version__)
print("Pandas:", pd.__version__)


Python: 3.12.13
Platform: Linux-6.17.0-1020-azure-x86_64-with-glibc2.39
GeoPandas: 1.1.4
Pandas: 3.0.5


## 1. Locate the processed 1 km grid


In [2]:
def find_project_root(start: Path) -> Path:
    """Find the repository root by locating the notebooks and Petropolis directories."""
    start = start.resolve()
    for candidate in [start, start.parent, *start.parents]:
        if (candidate / "notebooks").exists() and (candidate / "Petropolis").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")

PROJECT_ROOT = find_project_root(Path.cwd())

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "Petropolis"
    / "petropolis_grid_1km.parquet"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "intermediate"
    / "ohsome"
    / "Petropolis"
)

FULL_BOUNDARIES_FILE = OUTPUT_DIR / "petropolis_grid_1km_ohsome_boundaries.geojson"
TEST_BOUNDARIES_FILE = OUTPUT_DIR / "petropolis_grid_1km_ohsome_test_boundaries.geojson"
TEST_INDEX_FILE = OUTPUT_DIR / "petropolis_grid_1km_ohsome_test_cells.parquet"

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Processed 1 km grid not found: {INPUT_FILE}. "
        "Run Notebook 02 before executing this notebook."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input:", INPUT_FILE)
print("Output directory:", OUTPUT_DIR)


Project root: /home/runner/work/sotm-br-2026/sotm-br-2026
Input: /home/runner/work/sotm-br-2026/sotm-br-2026/data/processed/Petropolis/petropolis_grid_1km.parquet
Output directory: /home/runner/work/sotm-br-2026/sotm-br-2026/data/intermediate/ohsome/Petropolis


## 2. Load and validate the processed grid


In [3]:
grid_1km = gpd.read_parquet(INPUT_FILE)

required_columns = {
    "grid_1km_id",
    "source_method",
    "n_200m_cells",
    "cell_area_m2",
    "intersection_area_m2",
    "municipality_fraction",
    "spatial_relation",
    grid_1km.geometry.name,
}

missing_columns = sorted(required_columns.difference(grid_1km.columns))
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

checks = {
    "features_positive": len(grid_1km) > 0,
    "crs_defined": grid_1km.crs is not None,
    "crs_epsg_4674": grid_1km.crs is not None and grid_1km.crs.to_epsg() == 4674,
    "grid_id_non_null": grid_1km["grid_1km_id"].notna().all(),
    "grid_id_unique": grid_1km["grid_1km_id"].is_unique,
    "geometry_non_null": grid_1km.geometry.notna().all(),
    "geometry_non_empty": (~grid_1km.geometry.is_empty).all(),
    "geometry_valid": grid_1km.geometry.is_valid.all(),
    "all_intersect_petropolis": (
        grid_1km["spatial_relation"].isin(["within", "boundary_intersection"]).all()
    ),
    "municipality_fraction_bounded": (
        grid_1km["municipality_fraction"].between(0.0, 1.0).all()
    ),
}

display(pd.Series(checks, name="PASS"))
print(f"Features: {len(grid_1km):,}")
print("CRS:", grid_1km.crs)
print("Geometry types:")
display(grid_1km.geometry.geom_type.value_counts().to_frame("features"))

if not all(checks.values()):
    failed = [name for name, passed in checks.items() if not passed]
    raise ValueError(f"Input validation failed: {failed}")

print("PASS — Processed 1 km grid is valid.")


features_positive                True
crs_defined                      True
crs_epsg_4674                    True
grid_id_non_null                 True
grid_id_unique                   True
geometry_non_null                True
geometry_non_empty               True
geometry_valid                   True
all_intersect_petropolis         True
municipality_fraction_bounded    True
Name: PASS, dtype: bool

Features: 895
CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "SIRGAS 2000", "datum": {"type": "GeodeticReferenceFrame", "name": "Sistema de Referencia Geocentrico para las AmericaS 2000", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope": "Horizontal component of 3D system.", "area": "Latin America - Central America and South America - onshore and offshore. Brazil - onshore and offshore.", "bbox": {"south_latitude": -59.87, "west_longitude": -122.19, "north_latitude": 32.72, "east_longitude": -25.28}, "id": {"authority": "EPSG", "code": 4674}}
Geometry types:


,features
Polygon,559
MultiPolygon,336


PASS — Processed 1 km grid is valid.


## 3. Inspect analytical composition

Notebook 02 produced 895 complete 1 km cells intersecting Petrópolis. This section confirms the source-method composition and the municipal-intersection fractions before boundary preparation.


In [4]:
display(
    grid_1km["source_method"]
    .value_counts(dropna=False)
    .rename_axis("source_method")
    .reset_index(name="n_1km_cells")
)

display(
    grid_1km["spatial_relation"]
    .value_counts(dropna=False)
    .rename_axis("spatial_relation")
    .reset_index(name="n_1km_cells")
)

display(
    grid_1km["municipality_fraction"]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .to_frame("municipality_fraction")
)


,source_method,n_1km_cells
0,aggregated_200m,559
1,native_1km,336


,spatial_relation,n_1km_cells
0,within,682
1,boundary_intersection,213


,municipality_fraction
count,895.000000
mean,0.883960
std,0.273504
min,0.000169
1%,0.002666
5%,0.088131
25%,1.000000
50%,1.000000
75%,1.000000
95%,1.000000


## 4. Create the WGS 84 query geometry copy

The analytical GeoParquet remains in EPSG:4674. Only a copy used for API requests is transformed to **EPSG:4326**, as required by the ohsome API.

No geometry is clipped to the municipality. The `municipality_fraction` attribute continues to describe how much of each complete 1 km cell lies inside Petrópolis.


In [5]:
query_grid = grid_1km.to_crs("EPSG:4326").copy()

print("Analytical CRS:", grid_1km.crs)
print("Query CRS:", query_grid.crs)

query_checks = {
    "query_crs_epsg_4326": query_grid.crs.to_epsg() == 4326,
    "geometry_valid": query_grid.geometry.is_valid.all(),
    "geometry_non_empty": (~query_grid.geometry.is_empty).all(),
    "grid_id_unique": query_grid["grid_1km_id"].is_unique,
}

display(pd.Series(query_checks, name="PASS"))

if not all(query_checks.values()):
    failed = [name for name, passed in query_checks.items() if not passed]
    raise ValueError(f"WGS 84 query geometry validation failed: {failed}")


Analytical CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "SIRGAS 2000", "datum": {"type": "GeodeticReferenceFrame", "name": "Sistema de Referencia Geocentrico para las AmericaS 2000", "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattening": 298.257222101}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope": "Horizontal component of 3D system.", "area": "Latin America - Central America and South America - onshore and offshore. Brazil - onshore and offshore.", "bbox": {"south_latitude": -59.87, "west_longitude": -122.19, "north_latitude": 32.72, "east_longitude": -25.28}, "id": {"authority": "EPSG", "code": 4674}}
Query CRS: EPSG:4326


query_crs_epsg_4326    True
geometry_valid         True
geometry_non_empty     True
grid_id_unique         True
Name: PASS, dtype: bool

## 5. Validate longitude-latitude coordinate ranges


In [6]:
bounds = query_grid.geometry.bounds

coordinate_checks = {
    "minx_is_longitude": bounds["minx"].between(-180, 180).all(),
    "maxx_is_longitude": bounds["maxx"].between(-180, 180).all(),
    "miny_is_latitude": bounds["miny"].between(-90, 90).all(),
    "maxy_is_latitude": bounds["maxy"].between(-90, 90).all(),
}

display(pd.Series(coordinate_checks, name="PASS"))

print("Total WGS 84 bounds:")
print(tuple(query_grid.total_bounds))

if not all(coordinate_checks.values()):
    failed = [name for name, passed in coordinate_checks.items() if not passed]
    raise ValueError(f"Coordinate-range validation failed: {failed}")


minx_is_longitude    True
maxx_is_longitude    True
miny_is_latitude     True
maxy_is_latitude     True
Name: PASS, dtype: bool

Total WGS 84 bounds:
(np.float64(-43.38540198551317), np.float64(-22.574838513886213), np.float64(-42.975112805079824), np.float64(-22.195282551848326))


## 6. Build an ohsome-compatible identified GeoJSON FeatureCollection

For `bpolys`, the ohsome API accepts a GeoJSON `FeatureCollection`. Each feature is assigned:

```json
"properties": {"id": "<grid_1km_id>"}
```

This makes the permanent `grid_1km_id` available as the boundary identifier in `groupBy/boundary` responses.

Only the identifier is placed in the API boundary payload. Analytical attributes remain in the GeoParquet and can later be joined back by `grid_1km_id`.


In [7]:
def to_ohsome_feature_collection(gdf: gpd.GeoDataFrame) -> dict:
    """Convert a GeoDataFrame to an identified GeoJSON FeatureCollection for ohsome bpolys."""
    features = []
    for row in gdf[["grid_1km_id", gdf.geometry.name]].itertuples(index=False):
        grid_id = str(row[0])
        geom = row[1]
        features.append({
            "type": "Feature",
            "properties": {"id": grid_id},
            "geometry": mapping(geom),
        })

    return {
        "type": "FeatureCollection",
        "features": features,
    }


full_feature_collection = to_ohsome_feature_collection(query_grid)
full_payload = json.dumps(
    full_feature_collection,
    ensure_ascii=False,
    separators=(",", ":"),
)

print("FeatureCollection features:", len(full_feature_collection["features"]))
print("Serialized payload size:", f"{len(full_payload.encode('utf-8')) / 1024:.1f} KiB")
print("First boundary ID:", full_feature_collection["features"][0]["properties"]["id"])
print("First geometry type:", full_feature_collection["features"][0]["geometry"]["type"])


FeatureCollection features: 895
Serialized payload size: 627.4 KiB
First boundary ID: 1KME6094N8811
First geometry type: Polygon


## 7. Boundary-payload quality control

This section verifies that each FeatureCollection feature has exactly one stable ID, valid geometry, and no duplicate boundary identifier. It also reports payload size rather than assuming that all 895 cells should be sent in one request.

The next API-query notebook should batch boundaries conservatively and use POST requests for GeoJSON `bpolys`.


In [8]:
feature_ids = [
    feature["properties"].get("id")
    for feature in full_feature_collection["features"]
]

payload_checks = {
    "feature_count_matches_grid": len(feature_ids) == len(query_grid),
    "all_ids_present": all(feature_ids),
    "all_ids_unique": len(feature_ids) == len(set(feature_ids)),
    "ids_match_grid": set(feature_ids) == set(query_grid["grid_1km_id"].astype(str)),
    "all_geometries_present": all(
        feature.get("geometry") is not None
        for feature in full_feature_collection["features"]
    ),
}

display(pd.Series(payload_checks, name="PASS"))

if not all(payload_checks.values()):
    failed = [name for name, passed in payload_checks.items() if not passed]
    raise ValueError(f"Boundary-payload validation failed: {failed}")

print("PASS — Full boundary payload structure is valid.")


feature_count_matches_grid    True
all_ids_present               True
all_ids_unique                True
ids_match_grid                True
all_geometries_present        True
Name: PASS, dtype: bool

PASS — Full boundary payload structure is valid.


## 8. Create a deterministic 10-cell API test subset

The first live ohsome request should use a small and spatially representative subset rather than the complete 895-cell grid.

The test subset contains:

- 5 cells fully within Petrópolis;
- 5 boundary-intersection cells spanning the observed range of `municipality_fraction`.

This allows the next notebook to test both interior and peripheral 1 km cells while preserving complete cell geometries.


In [9]:
def select_evenly_spaced_rows(df: pd.DataFrame, n: int, sort_column: str) -> pd.DataFrame:
    """Select up to n rows approximately evenly across a sorted numeric variable."""
    ordered = df.sort_values([sort_column, "grid_1km_id"]).reset_index(drop=True)
    if len(ordered) <= n:
        return ordered.copy()

    positions = np.linspace(0, len(ordered) - 1, n)
    indices = np.unique(np.round(positions).astype(int))

    if len(indices) < n:
        remaining = [i for i in range(len(ordered)) if i not in set(indices)]
        indices = np.concatenate([indices, remaining[: n - len(indices)]])

    return ordered.iloc[np.sort(indices[:n])].copy()


within_candidates = query_grid[
    query_grid["spatial_relation"].eq("within")
].copy()

boundary_candidates = query_grid[
    query_grid["spatial_relation"].eq("boundary_intersection")
].copy()

test_within = within_candidates.sort_values("grid_1km_id").head(5).copy()
test_boundary = select_evenly_spaced_rows(
    boundary_candidates,
    n=5,
    sort_column="municipality_fraction",
)

test_grid = (
    pd.concat([test_within, test_boundary], ignore_index=True)
    .pipe(
        lambda df: gpd.GeoDataFrame(
            df,
            geometry=query_grid.geometry.name,
            crs=query_grid.crs,
        )
    )
)

if len(test_grid) != 10:
    raise ValueError(
        f"Expected a 10-cell test subset but selected {len(test_grid)} cells."
    )

if not test_grid["grid_1km_id"].is_unique:
    raise ValueError("Test subset contains duplicate grid IDs.")

display(
    test_grid[
        [
            "grid_1km_id",
            "source_method",
            "n_200m_cells",
            "municipality_fraction",
            "spatial_relation",
        ]
    ].sort_values(["spatial_relation", "municipality_fraction"])
)


,grid_1km_id,source_method,n_200m_cells,municipality_fraction,spatial_relation
5,1KME6120N8829,aggregated_200m,25,0.000169,boundary_intersection
6,1KME6118N8826,aggregated_200m,25,0.152368,boundary_intersection
7,1KME6107N8797,aggregated_200m,25,0.557120,boundary_intersection
8,1KME6097N8811,native_1km,0,0.880925,boundary_intersection
9,1KME6119N8801,aggregated_200m,25,0.999933,boundary_intersection
0,1KME6096N8812,native_1km,0,1.000000,within
2,1KME6097N8813,native_1km,0,1.000000,within
3,1KME6098N8812,native_1km,0,1.000000,within
4,1KME6098N8813,native_1km,0,1.000000,within
1,1KME6097N8812,native_1km,0,1.000000,within


## 9. Build and validate the test `bpolys` FeatureCollection


In [10]:
test_feature_collection = to_ohsome_feature_collection(test_grid)

test_payload = json.dumps(
    test_feature_collection,
    ensure_ascii=False,
    separators=(",", ":"),
)

test_ids = [
    feature["properties"]["id"]
    for feature in test_feature_collection["features"]
]

test_checks = {
    "exactly_10_boundaries": len(test_ids) == 10,
    "unique_ids": len(test_ids) == len(set(test_ids)),
    "ids_match_test_grid": set(test_ids) == set(test_grid["grid_1km_id"].astype(str)),
    "valid_geometries": test_grid.geometry.is_valid.all(),
    "query_crs_epsg_4326": test_grid.crs.to_epsg() == 4326,
}

display(pd.Series(test_checks, name="PASS"))
print("Test payload size:", f"{len(test_payload.encode('utf-8')) / 1024:.1f} KiB")
print("Test boundary IDs:")
for grid_id in test_ids:
    print(" -", grid_id)

if not all(test_checks.values()):
    failed = [name for name, passed in test_checks.items() if not passed]
    raise ValueError(f"Test-boundary validation failed: {failed}")


exactly_10_boundaries    True
unique_ids               True
ids_match_test_grid      True
valid_geometries         True
query_crs_epsg_4326      True
Name: PASS, dtype: bool

Test payload size: 5.6 KiB
Test boundary IDs:
 - 1KME6096N8812
 - 1KME6097N8812
 - 1KME6097N8813
 - 1KME6098N8812
 - 1KME6098N8813
 - 1KME6120N8829
 - 1KME6118N8826
 - 1KME6107N8797
 - 1KME6097N8811
 - 1KME6119N8801


## 10. Save the prepared boundary files

Three intermediate files are written:

1. the full 895-cell identified GeoJSON FeatureCollection;
2. the deterministic 10-cell test FeatureCollection;
3. the corresponding 10-cell GeoParquet index with analytical attributes.

The GeoJSON files are API-ready spatial inputs. The test GeoParquet preserves the analytical metadata needed to interpret the API responses.


In [11]:
with FULL_BOUNDARIES_FILE.open("w", encoding="utf-8") as f:
    json.dump(
        full_feature_collection,
        f,
        ensure_ascii=False,
        separators=(",", ":"),
    )

with TEST_BOUNDARIES_FILE.open("w", encoding="utf-8") as f:
    json.dump(
        test_feature_collection,
        f,
        ensure_ascii=False,
        separators=(",", ":"),
    )

test_grid.to_parquet(TEST_INDEX_FILE, index=False)

print("Saved:", FULL_BOUNDARIES_FILE)
print("Saved:", TEST_BOUNDARIES_FILE)
print("Saved:", TEST_INDEX_FILE)


Saved: /home/runner/work/sotm-br-2026/sotm-br-2026/data/intermediate/ohsome/Petropolis/petropolis_grid_1km_ohsome_boundaries.geojson
Saved: /home/runner/work/sotm-br-2026/sotm-br-2026/data/intermediate/ohsome/Petropolis/petropolis_grid_1km_ohsome_test_boundaries.geojson
Saved: /home/runner/work/sotm-br-2026/sotm-br-2026/data/intermediate/ohsome/Petropolis/petropolis_grid_1km_ohsome_test_cells.parquet


## 11. Prepare the next request without sending it

The next notebook should begin with a small live request such as road length grouped by boundary.

The current ohsome API documentation recommends the `filter` parameter rather than deprecated `types`, `keys`, and `values` parameters. A broad road-network starting filter can be:

```text
highway=* and geometry:line
```

A POST request can then send the 10-cell GeoJSON FeatureCollection through `bpolys` to:

```text
https://api.ohsome.org/v1/elements/length/groupBy/boundary
```

No HTTP request is sent in this notebook.


In [12]:
OHSOME_BASE_URL = "https://api.ohsome.org/v1"
TEST_ENDPOINT = f"{OHSOME_BASE_URL}/elements/length/groupBy/boundary"

request_preview = {
    "method": "POST",
    "url": TEST_ENDPOINT,
    "form_fields": {
        "bpolys": "<serialized GeoJSON FeatureCollection from test_payload>",
        "filter": "highway=* and geometry:line",
        "format": "json",
        "time": "<to be defined in Notebook 04 after reading /metadata>",
    },
    "boundary_count": len(test_feature_collection["features"]),
    "payload_size_bytes": len(test_payload.encode("utf-8")),
}

display(pd.Series(request_preview, name="request_preview"))


method                                                                                                                                   POST
url                                                                                https://api.ohsome.org/v1/elements/length/groupBy/boundary
form_fields           {'bpolys': '<serialized GeoJSON FeatureCollection from test_payload>', 'filter': 'highway=* and geometry:line', 'for...
boundary_count                                                                                                                             10
payload_size_bytes                                                                                                                       5782
Name: request_preview, dtype: object

## 12. PASS / WARNING / FAIL summary


In [13]:
def diagnostic_status(condition: bool, warning: bool = False) -> str:
    if condition:
        return "PASS"
    return "WARNING" if warning else "FAIL"


full_payload_kib = len(full_payload.encode("utf-8")) / 1024
test_payload_kib = len(test_payload.encode("utf-8")) / 1024

diagnostics = [
    [
        "Processed grid contains 895 cells",
        diagnostic_status(len(grid_1km) == 895, warning=len(grid_1km) > 0),
        f"{len(grid_1km):,}",
    ],
    [
        "Permanent grid IDs are unique",
        diagnostic_status(query_grid["grid_1km_id"].is_unique),
        f"{query_grid['grid_1km_id'].nunique():,} unique IDs",
    ],
    [
        "Query geometries use EPSG:4326",
        diagnostic_status(query_grid.crs.to_epsg() == 4326),
        str(query_grid.crs),
    ],
    [
        "All query geometries are valid",
        diagnostic_status(query_grid.geometry.is_valid.all()),
        f"{(~query_grid.geometry.is_valid).sum():,} invalid",
    ],
    [
        "All coordinate ranges are valid lon-lat",
        diagnostic_status(all(coordinate_checks.values())),
        str(tuple(query_grid.total_bounds)),
    ],
    [
        "Full FeatureCollection matches the grid",
        diagnostic_status(all(payload_checks.values())),
        f"{len(full_feature_collection['features']):,} boundaries",
    ],
    [
        "10-cell test subset is valid",
        diagnostic_status(all(test_checks.values())),
        f"{len(test_grid)} boundaries",
    ],
    [
        "Test payload is compact",
        diagnostic_status(test_payload_kib < 100, warning=True),
        f"{test_payload_kib:.1f} KiB",
    ],
    [
        "Full payload size recorded for batching decisions",
        "PASS",
        f"{full_payload_kib:.1f} KiB",
    ],
]

diagnostic_table = pd.DataFrame(
    diagnostics,
    columns=["check", "status", "value"],
)

display(diagnostic_table)
display(
    diagnostic_table["status"]
    .value_counts()
    .reindex(["PASS", "WARNING", "FAIL"], fill_value=0)
    .to_frame("count")
)

if (diagnostic_table["status"] == "FAIL").any():
    raise ValueError("Boundary preparation contains one or more FAIL diagnostics.")

print("PASS — ohsome boundary preparation is ready for the first live API test.")


,check,status,value
0,Processed grid contains 895 cells,PASS,895
1,Permanent grid IDs are unique,PASS,895 unique IDs
2,Query geometries use EPSG:4326,PASS,EPSG:4326
3,All query geometries are valid,PASS,0 invalid
4,All coordinate ranges are valid lon-lat,PASS,"(np.float64(-43.38540198551317), np.float64(-22.574838513886213), np.float64(-42.975112805079824), np.float64(-22.19..."
5,Full FeatureCollection matches the grid,PASS,895 boundaries
6,10-cell test subset is valid,PASS,10 boundaries
7,Test payload is compact,PASS,5.6 KiB
8,Full payload size recorded for batching decisions,PASS,627.4 KiB


,count
status,
PASS,9
WARNING,0
FAIL,0


PASS — ohsome boundary preparation is ready for the first live API test.


## 13. Next step

Notebook 04 should **not** start with all 895 cells.

Recommended sequence:

1. query `GET /metadata` to record the current ohsome API/OSHDB temporal extent;
2. use the 10-cell test FeatureCollection;
3. test `/elements/length/groupBy/boundary` with `highway=* and geometry:line`;
4. verify returned boundary IDs and values;
5. test `/elements/count/groupBy/boundary`;
6. test `/users/count/groupBy/boundary`;
7. test `/contributions/count/groupBy/boundary`;
8. only after semantic validation, define batching, caching, retry logic, and the full temporal extraction strategy.

The analytical unit remains the complete official/reconstructed IBGE 1 km cell identified by `grid_1km_id`.
